In [80]:
import numpy as np
import joblib
from sklearn.ensemble import RandomForestRegressor


data = joblib.load("rf_model.pkl")
rf = data["rf"]

#Update: results from SPT+Planck+ACT+DESI https://arxiv.org/pdf/2506.20707 Table 6
#P-ACT results https://arxiv.org/pdf/2503.14452 Table 5
yobs = np.array([3.0574,0.9728,-0.0045])
yerr = np.array([0.0094,0.0027,0.0067])

# =========================================================
# ADAPTIVE LIKELIHOOD FUNCTION
# =========================================================

def logp(phi0,gst,V0,Q0):

    theta = np.array([phi0, gst, V0, Q0], dtype=float)

    theta = np.log10(theta)

    preds = np.array([
        tree.predict(theta.reshape(1, -1))[0]
        for tree in rf.estimators_
    ])

    mean_pred = np.mean(preds, axis=0)
    std_pred  = np.std(preds, axis=0)

    # Use only required observables
    model_fin = mean_pred[:len(yobs)]

    # ---------------------------------
    # Likelihood (with RF uncertainty)
    # ---------------------------------
    sigma2 = (yerr**2) 

    log_lik = -0.5 * np.sum(
        np.log(2*np.pi*sigma2) +
        ((yobs - model_fin)**2) / sigma2
    )

    # ---------------------------------
    # Debug (optional)
    # --------------------------np.all(std_pred[:len(yobs)] < 2*(yerr + yerr_solver))-------
    print("theta (log10):", theta)
    print("prediction:", model_fin)
    print("RF std:", std_pred[:len(yobs)])
    print("logL:", log_lik)
    print("--------------------------")

    return log_lik

In [81]:
logp(4.91707,8.90846,1.34913e-13,60.52286)

theta (log10): [  0.69170639   0.94980263 -12.8699462    1.78191944]
prediction: [ 3.08342448e+00  9.72363611e-01 -7.54718160e-04]
RF std: [0.06593064 0.00989407 0.00258284]
logL: 8.828615042900246
--------------------------


np.float64(8.828615042900246)

In [82]:
scale = yerr + [1,1,1]
rel_std = [2.89824621e+00, 9.62918398e-01, 1.15662653e-03] / scale
score = np.max(rel_std)
print(score)

2.8712563998414895
